In [1]:
import torch
import numpy as np

Nous allons voir comment la matrice de transition se construit concrètement à partir de particules labélisées(qui appartiennent chacune à une partition déjà définie)

Supposons que nous disposons dans le mélangeur partitionné en 3 partitions sept particules dont les états précedent et actuel sont representés par les vecteurs d'états **s_prev** et **s_curr**respectivement

In [2]:
s_prev=torch.tensor([0,0,1,2,1,0,0])
s_curr=torch.tensor([0,1,2,0,1,1,1])

Nous allons maintenant construire la fonction $\phi$ qui determine si oui ou non une particule se retrouve dans une partition donnée

In [3]:
n_states=3
device="cpu"
phi_prev = (s_prev.unsqueeze(1) == torch.arange(n_states, device=device)).float()  # (n, n_states)
phi_curr = (s_curr.unsqueeze(1) == torch.arange(n_states, device=device)).float()  # (n, n_states)

In [4]:
phi_prev


tensor([[1., 0., 0.],
        [1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [0., 1., 0.],
        [1., 0., 0.],
        [1., 0., 0.]])

In [5]:
phi_curr

tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.]])

Nous constatons que les matrices **phi** sont telles que les lignes representent les particules qui se retrouvent dans une et une seule partition donnée, suivant les colonnes de la matrice.

On voit donc qu'à l'instant précedent, la particule numéro 1 (premier élément de la liste ou encore particule d'ince 0) se trouve dans la première partition(d'indice O), la deuxième particule dans la premi!re partition également et ainsi de suite.

À l'instant present, nous constatons que la particule d'indice 0 qui était dans la partition d'indice 0 reste dans cette partition; et que la particule d'indice 1 qui était dans la partition d'indice 0 se retrouve dans la partion d'indice 1. 

C'est ainsi que se construit ces deux matrices phi qui chaque ligne représente une particule et chaque colonne représente la partition dans laquelle elle se trouve.

In [6]:
transition=phi_prev.T@phi_curr


In [7]:
transition


tensor([[1., 3., 0.],
        [0., 1., 1.],
        [1., 0., 0.]])

La matrice de transition est donc une matrice qui représente les partitions du mélangeur entre deux instants et   est  telle que chaque ligne represente l'instant précedent du mélangeur et chaque colonne représente l'instant present.

Autrement dit, lorsque l'on regarde la première ligne de la matrice, sa somme représente le nombre de particules qui étaient à l'instant précedent dans la partition d'indice 0: soit 4 particules .Ces particules vont chacune se déplacer pour aller dans une autre partition à l'instant suivant:
Une ira dans la partition d'indice 0 et 3 iront dans la partition d'indice 1.


 La somme de la deuxième ligne représente le nombre de particule qui étaient à l'instant précedent dans la partition d'indice 1: soit 2 particules; dont une ira dans la partition d'indice 1 et une autre ira dans la partition d'ince 2.
 
et la troisième ligne représente le nombre de particule qui étaient à l'instant précedent dans la partition d'indice 2: soit 1 particule, qui ira dans la partition d'indice 0.

Il en est de même pour les colonnes dont la somme représente le nombre de particules qui sont à l'instant présent dans une partition donnéé: la partion d'incide 0 possède 2 particules à l'instant présent, tandis que la partition d'indice 1 possède 4 particules et la partition d'indice 2 en possède 1 particule à l'instant présent.

Maintenant, voyons comment les particules se sont réparties:
La partition d'indice 0 possedait 4 particules qui à l'instant present possède que 2 qui proviennent des partitions d'indice 0 et 2 de l'instant précedent. Pareillement la partition d'indice 1 possedait 2 particules à l'instant précedent et maintenant possède 4 particules dont 3 proviennent de la partition d'indice 0 de l'instant précédent et 1 de la partition d'indice 1 de l'instant précedent. La partition d'indice 2 qui possedait un particule possède toujours une particule mais qui provient de la partition d'indice 1.

On se rend donc compte que l'on peut lire le passé et le futur dans la matrice de transition telle que construite.



In [8]:
transition[:]=transition.T/phi_prev.sum(0)
transition


tensor([[0.2500, 0.0000, 1.0000],
        [0.7500, 0.5000, 0.0000],
        [0.0000, 0.5000, 0.0000]])

In [9]:
transition[:]=transition/phi_curr.sum(0,keepdim=True)
transition


tensor([[0.1250, 0.0000, 1.0000],
        [0.3750, 0.1250, 0.0000],
        [0.0000, 0.1250, 0.0000]])

In [10]:
print(f"ID avant : {id(transition)}")
# Votre opération
transition = transition.T / phi_prev.sum(0)
print(f"ID après : {id(transition)}")

ID avant : 139974767867088
ID après : 139973991581088


In [11]:
s_predict=phi_prev.sum(0)@transition

In [12]:
s_predict

tensor([0.3750, 0.8750, 0.2500])

In [13]:
phi_curr.sum(0)

tensor([2., 4., 1.])

Vérifions le code qui calcule la matrice de transition dans le fichier principal

In [21]:
def compute_P_matrix_torch(states_prev, states_curr, n_states, device="cpu"):
    """
    Calcule P_n pour un timestep - version entièrement vectorisée.
    P[j,i] = probabilité de transition de l'état i vers l'état j
    """
    # Conversion en tensor si nécessaire
    if isinstance(states_prev, np.ndarray):
        states_prev = torch.from_numpy(states_prev)
    if isinstance(states_curr, np.ndarray):
        states_curr = torch.from_numpy(states_curr)
    
    s_prev = states_prev.to(device).long()
    s_curr = states_curr.to(device).long()
    
    n = min(len(s_prev), len(s_curr))
    s_prev = s_prev[:n]
    s_curr = s_curr[:n]
    
    # Création des masques one-hot pour chaque particule
    # phi_prev[p, i] = 1 si particule p était dans état i
    # phi_curr[p, j] = 1 si particule p est dans état j
    phi_prev = (s_prev.unsqueeze(1) == torch.arange(n_states, device=device)).float()  # (n, n_states)
    phi_curr = (s_curr.unsqueeze(1) == torch.arange(n_states, device=device)).float()  # (n, n_states)
    
    # Matrice de co-occurrence : transitions[i, j] = nombre de transitions i → j
    # Somme sur toutes les particules de phi_prev[:, i] * phi_curr[:, j]
    transitions = phi_prev.T @ phi_curr  # (n_states, n_states)
    
    # Dénominateur : nombre de particules dans chaque état au temps précédent
    denominator = phi_prev.sum(dim=0)  # (n_states,)
    
    # P[i, j] = transitions[i, j] / denominator[i]
    # P = transitions / denominator.unsqueeze(1).clamp(min=1e-10)
    P=transitions.T/denominator
    
    # Mettre à zéro les lignes sans particules
    P[denominator == 0] = 0.0
    
    # Transposition pour avoir P[j, i] = prob(i → j)
    # P = P.T
    
    return P.to(torch.float64)

In [30]:
 P = torch.zeros((3,3), dtype=torch.float64, device="cpu")
for i in range(3):
    P+=compute_P_matrix_torch(s_prev,s_curr,3)
P/3

tensor([[0.2500, 0.0000, 1.0000],
        [0.7500, 0.5000, 0.0000],
        [0.0000, 0.5000, 0.0000]], dtype=torch.float64)

In [34]:
1//10

0